In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("results.csv")

required = ["mode", "error_code"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"CSV não tem colunas mínimas {missing}. Colunas atuais: {list(df.columns)}")

for c in ["t_seed_ms", "t_kem_ms", "t_total_ms"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["mode"] = df["mode"].astype(str).str.strip().str.lower()
df["ok"] = df["error_code"].astype(str).str.upper().eq("OK")

ok = df[df["ok"]].copy()


In [3]:
def bootstrap_ci(x, stat=np.median, n_boot=5000, alpha=0.05, seed=42):
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    boots = rng.choice(x, size=(n_boot, len(x)), replace=True)
    stats = np.apply_along_axis(stat, 1, boots)
    lo = np.quantile(stats, alpha/2)
    hi = np.quantile(stats, 1 - alpha/2)
    return float(lo), float(hi)

In [4]:
# -----------------------
# 2) Gráfico A: barras (t_total_ms por mode)
# -----------------------
modes_order = [m for m in ["baseline", "eaas", "direct"] if m in ok["mode"].unique()]

rows = []
for m in modes_order:
    x = ok.loc[(ok["mode"] == m) & ok["t_total_ms"].notna(), "t_total_ms"].to_numpy()
    med = float(np.median(x)) if len(x) else np.nan
    lo, hi = bootstrap_ci(x, stat=np.median)
    rows.append({
        "mode": m,
        "t_total_ms_median": med,
        "t_total_ms_ci95_lo": lo,
        "t_total_ms_ci95_hi": hi,
        "n_ok": int(len(x)),
    })

bar_total = pd.DataFrame(rows)

In [5]:
# -----------------------
# 3) Gráfico B: barras empilhadas (medianas t_seed_ms e t_kem_ms)
# -----------------------
modes_decomp = [m for m in ["eaas", "direct"] if m in ok["mode"].unique()]

bar_decomp = (
    ok[ok["mode"].isin(modes_decomp)]
    .groupby("mode")[["t_seed_ms", "t_kem_ms"]]
    .median()
    .rename(columns={"t_seed_ms": "t_seed_ms_median", "t_kem_ms": "t_kem_ms_median"})
    .reset_index()
)

# mantém ordem eaas, direct
bar_decomp["mode"] = pd.Categorical(bar_decomp["mode"], categories=modes_decomp, ordered=True)
bar_decomp = bar_decomp.sort_values("mode").reset_index(drop=True)

In [6]:
bar_total.to_csv("bar_total_latency.csv", index=False)
bar_decomp.to_csv("bar_decomp.csv", index=False)

print("Gerados: bar_total_latency.csv e bar_decomp.csv")
print(bar_total)
print(bar_decomp)


Gerados: bar_total_latency.csv e bar_decomp.csv
       mode  t_total_ms_median  t_total_ms_ci95_lo  t_total_ms_ci95_hi  n_ok
0  baseline             7.9570              7.7655               8.153   100
1      eaas            73.1290             72.1540              74.291   100
2    direct          1214.4025           1212.7400            1215.433   100
     mode  t_seed_ms_median  t_kem_ms_median
0    eaas            64.921           8.2140
1  direct          1206.225           8.5545
